# Basic Prompt Agent (Default Project) — `@azure/ai-projects`

This notebook demonstrates basic Prompt Agent operations using the `AIProjectClient` against a **default project endpoint**: create an agent, create a conversation, generate responses, and clean up resources.

It mirrors the [`agentBasicWithDefaultProject.ts`](./agentBasicWithDefaultProject.ts) sample and runs the **locally built** `@azure/ai-projects` from this repo.

## Prerequisites

1. **Build the package first** so `dist/` is current: `cd sdk/ai/ai-projects && pnpm build`
2. **tslab kernel** installed and registered (`npm install -g tslab` then `tslab install`); select the **TypeScript** (tslab) kernel.
3. **Launch VS Code / Jupyter from `sdk/ai/ai-projects/`** so Node resolves the local `@azure/ai-projects`.
4. **`az login`** completed so `DefaultAzureCredential` can authenticate.
5. **Environment variables**: `FOUNDRY_PROJECT_DEFAULT_ENDPOINT`, `FOUNDRY_MODEL_NAME`.

Run the cells in order (top to bottom); state is shared across cells.

In [1]:
// Imports and configuration
import { DefaultAzureCredential } from "@azure/identity";
import { AIProjectClient } from "@azure/ai-projects";

const projectEndpoint = process.env["FOUNDRY_PROJECT_DEFAULT_ENDPOINT"] ?? "<project endpoint>";
const deploymentName = process.env["FOUNDRY_MODEL_NAME"] ?? "<model deployment name>";

console.log(`Model: ${deploymentName}`);

Model: gpt-5.2


In [2]:
// Create the AI Project client and the OpenAI client
const project = new AIProjectClient(projectEndpoint, new DefaultAzureCredential());
// Annotated as `any` so tslab does not try to emit a non-portable declaration
// referencing the deep `node_modules/openai` (pnpm junction) path.
const openAIClient: any = project.getOpenAIClient();

In [3]:
// Create the agent
console.log("Creating agent...");
const agent = await project.agents.createVersion("my-agent-basic", {
  kind: "prompt",
  model: deploymentName,
  instructions: "You are a helpful assistant that answers general questions",
});
console.log(`Agent created (id: ${agent.id}, name: ${agent.name}, version: ${agent.version})`);

Creating agent...
Agent created (id: my-agent-basic:31, name: my-agent-basic, version: 31)


In [4]:
// Create a conversation with an initial user message
console.log("Creating conversation with initial user message...");
const conversation = await openAIClient.conversations.create({
  items: [
    { type: "message", role: "user", content: "What is the size of France in square miles?" },
  ],
});
console.log(`Created conversation with initial user message (id: ${conversation.id})`);

Creating conversation with initial user message...
Created conversation with initial user message (id: conv_6e67abc5a2809aea00XhOSasBsndz8XG1NI7kqj9Je2cfc3rj1)


In [5]:
// Generate the first response using the agent
console.log("Generating response...");
const franceResponse = await openAIClient.responses.create(
  {
    conversation: conversation.id,
  },
  {
    body: { agent_reference: { name: agent.name, type: "agent_reference" } },
  },
);
console.log(`Response output: ${franceResponse.output_text}`);

Generating response...


Response output: France has an area of about **211,000 square miles** (≈ **551,700 square kilometers**) for **metropolitan France** (mainland + Corsica). If you include overseas territories, **France totals about 248,600 square miles** (≈ **643,800 square kilometers**).


In [6]:
// Add a second user message to the conversation
console.log("Adding a second user message to the conversation...");
await openAIClient.conversations.items.create(conversation.id, {
  items: [{ type: "message", role: "user", content: "And what is the capital city?" }],
});
console.log("Added a second user message to the conversation");

Adding a second user message to the conversation...


Added a second user message to the conversation


In [7]:
// Generate the second response
console.log("Generating second response...");
const capitalResponse = await openAIClient.responses.create(
  {
    conversation: conversation.id,
  },
  {
    body: { agent_reference: { name: agent.name, type: "agent_reference" } },
  },
);
console.log(`Response output: ${capitalResponse.output_text}`);

Generating second response...
Response output: The capital city of France is **Paris**.


In [8]:
// Clean up resources
console.log("Cleaning up resources...");
await openAIClient.conversations.delete(conversation.id);
console.log("Conversation deleted");

await project.agents.deleteVersion(agent.name, agent.version);
console.log("Agent deleted");

Cleaning up resources...
Conversation deleted
Agent deleted
